In [12]:
!pip install nba_api

In [13]:
# Cell 1 is for the import statements and the setup for the project
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from nba_api.stats.endpoints import leaguegamefinder
from nba_api.stats.static import teams
import time


In [14]:
# Get the results across the past 5 seasons to compare to the current season at the end for "accuracy" in a way
seasons = ['2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
all_games= []

for season in seasons:
  print(f"Pulling seasons: {season}")
  gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable=season)
  games = gamefinder.get_data_frames()[0]
  all_games.append(games)
  time.sleep(2)

df = pd.concat(all_games, ignore_index=True)
print(f"\nTotal rows: {len(df)}")
print(df.columns.tolist())
print(df.head())

# Following this output, each row is one team's performance in one game, so every game has two rows (one per team). The columns display the stats of a team points,
# FG%, assists, steals, rebounds, blocks, turnovers, plus/minus, and win/loss. This is the skeleton of a dataset we will use to build features and train models.

Pulling seasons: 2019-20
Pulling seasons: 2020-21
Pulling seasons: 2021-22
Pulling seasons: 2022-23
Pulling seasons: 2023-24
Pulling seasons: 2024-25

Total rows: 16133
['SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'MIN', 'PTS', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PLUS_MINUS']
  SEASON_ID     TEAM_ID TEAM_ABBREVIATION           TEAM_NAME     GAME_ID  \
0     42019  1610612747               LAL  Los Angeles Lakers  0041900406   
1     42019  1610612748               MIA          Miami Heat  0041900406   
2     42019  1610612748               MIA          Miami Heat  0041900405   
3     42019  1610612747               LAL  Los Angeles Lakers  0041900405   
4     42019  1610612748               MIA          Miami Heat  0041900404   

    GAME_DATE      MATCHUP WL  MIN  PTS  ...  FT_PCT  OREB  DREB  REB  AST  \
0  2020-10-11    LAL @ MIA  W 

In [15]:
# figuring out the dataframe shape, what datatypes are each column, and figuring out if there are missing values I will need to deal with

print(df.shape)
print(df.dtypes)
print(df.isnull().sum)

(16133, 28)
SEASON_ID             object
TEAM_ID                int64
TEAM_ABBREVIATION     object
TEAM_NAME             object
GAME_ID               object
GAME_DATE             object
MATCHUP               object
WL                    object
MIN                    int64
PTS                    int64
FGM                    int64
FGA                    int64
FG_PCT               float64
FG3M                   int64
FG3A                   int64
FG3_PCT              float64
FTM                    int64
FTA                    int64
FT_PCT               float64
OREB                   int64
DREB                   int64
REB                    int64
AST                    int64
STL                    int64
BLK                    int64
TOV                    int64
PF                     int64
PLUS_MINUS           float64
dtype: object
<bound method DataFrame.sum of        SEASON_ID  TEAM_ID  TEAM_ABBREVIATION  TEAM_NAME  GAME_ID  GAME_DATE  \
0          False    False              False      Fa

In [16]:
# Create HOME and WIN columns, convert date to datatime, sort data
df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'])
df['HOME'] = df['MATCHUP'].apply(lambda x: 1 if "vs." in x else 0)
df['WIN'] = df['WL'].apply(lambda x: 1 if x == 'W' else 0)
df = df.sort_values(['TEAM_ID', 'GAME_DATE']).reset_index(drop=True)
print("Cleaning complete.")

Cleaning complete.


In [17]:
# DATA VERIFICATION - cross check against known real stats


# GSW 2021-22: real record was 53-29, ~111 PPG
gsw_2122 = df[
    (df['TEAM_ABBREVIATION'] == 'GSW') &
    (df['SEASON_ID'] == '22021')
]
print(f"GSW 2021-22 games: {len(gsw_2122)} (should be 82)")
print(f"GSW 2021-22 avg PTS: {gsw_2122['PTS'].mean():.1f} (should be ~111)")
print(f"GSW 2021-22 wins: {gsw_2122['WIN'].sum()} (should be 53)")
print(f"GSW 2021-22 losses: {(gsw_2122['WIN']==0).sum()} (should be 29)")
print()

# BOS 2022-23: real record was 57-25, ~117 PPG
bos_2223 = df[
    (df['TEAM_ABBREVIATION'] == 'BOS') &
    (df['SEASON_ID'] == '22022')
]
print(f"BOS 2022-23 games: {len(bos_2223)} (should be 82)")
print(f"BOS 2022-23 avg PTS: {bos_2223['PTS'].mean():.1f} (should be ~117)")
print(f"BOS 2022-23 wins: {bos_2223['WIN'].sum()} (should be 57)")
print(f"BOS 2022-23 losses: {(bos_2223['WIN']==0).sum()} (should be 25)")
print()

# LAL 2023-24: real record was 47-35, ~114 PPG
lal_2324 = df[
    (df['TEAM_ABBREVIATION'] == 'LAL') &
    (df['SEASON_ID'] == '22023')
]
print(f"LAL 2023-24 games: {len(lal_2324)} (should be 82)")
print(f"LAL 2023-24 avg PTS: {lal_2324['PTS'].mean():.1f} (should be ~1183)")
print(f"LAL 2023-24 wins: {lal_2324['WIN'].sum()} (should be 47)")
print(f"LAL 2023-24 losses: {(lal_2324['WIN']==0).sum()} (should be 35)")


#Okay so based on these outputs and comparing to real life stats

GSW 2021-22 games: 82 (should be 82)
GSW 2021-22 avg PTS: 111.0 (should be ~111)
GSW 2021-22 wins: 53 (should be 53)
GSW 2021-22 losses: 29 (should be 29)

BOS 2022-23 games: 82 (should be 82)
BOS 2022-23 avg PTS: 117.9 (should be ~117)
BOS 2022-23 wins: 57 (should be 57)
BOS 2022-23 losses: 25 (should be 25)

LAL 2023-24 games: 82 (should be 82)
LAL 2023-24 avg PTS: 118.0 (should be ~114)
LAL 2023-24 wins: 47 (should be 47)
LAL 2023-24 losses: 35 (should be 35)
